# 04 — Извлечь учебные PDF и добавить в Qdrant

Ноутбук обрабатывает PDF из `data/KUBSU_DOCS` отдельно от Telegram-переписки и добавляет их чанки в уже существующую коллекцию `telegram_keep_user2_350tok_50overlap`. Поиск в Streamlit увидит документы автоматически.

Текст читается из PDF штатным слоем `pdftotext`, без OCR. Перед индексацией создаются постраничный CSV и отчёт качества: страницы без текста и короткие артефакты не попадают в чанки. Все индексированные чанки сохраняют абсолютный путь к исходному PDF.

Размер чанка — 350 токенов USER2-base, перекрытие — 50 токенов. Повторный запуск обновляет только точки с `source_group=kubsu_docs`; Telegram-точки не затрагиваются. Сначала выполните ячейки извлечения и проверьте отчёт, затем запустите ячейки чанкинга и индексации.

In [ ]:
# При необходимости установите зависимости в kernel этого ноутбука:
# %pip install "sentence-transformers==5.2.2" "transformers==4.57.4" "qdrant-client>=1.19" pandas tqdm

In [ ]:
from __future__ import annotations

import hashlib
import re
import shutil
import subprocess
import unicodedata
import uuid
from pathlib import Path

import pandas as pd
from IPython.display import display
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "data").is_dir()
)
PDF_DIR = ROOT / "data" / "KUBSU_DOCS"
OUTPUT_DIR = ROOT / "data" / "processed" / "kubsu_docs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PDF_FILES = sorted(PDF_DIR.rglob("*.pdf"), key=lambda path: path.name.casefold())

BUNDLED_PDFTOTEXT = (
    Path.home()
    / ".cache/codex-runtimes/codex-primary-runtime/dependencies/native/poppler/poppler/bin/pdftotext"
)
PDFTOTEXT = shutil.which("pdftotext") or (
    str(BUNDLED_PDFTOTEXT) if BUNDLED_PDFTOTEXT.is_file() else None
)
assert PDF_DIR.is_dir(), f"Не найдена папка с документами: {PDF_DIR}"
assert PDF_FILES, f"В {PDF_DIR} не найдено PDF-файлов"
assert PDFTOTEXT, "Не найден pdftotext. Установите Poppler и перезапустите kernel."

QDRANT_URL = "http://localhost:6333"
COLLECTION = "telegram_keep_user2_350tok_50overlap"
MODEL_NAME = "deepvk/USER2-base"
CHUNK_SIZE_TOKENS = 350
CHUNK_OVERLAP_TOKENS = 50
BATCH_SIZE = 64
MIN_PAGE_CHARACTERS = 100
SOURCE_GROUP = "kubsu_docs"

# Cover page in this file is a scanned image; add its verified title to every chunk.
MANUAL_TITLES = {
    "RPD_OVP_2025.pdf": "Основы военной подготовки",
    "B1.O.02 Annotacii Pravovedenie 09.03.02 Informacionnye sistemy i tehnologii.pdf": "Правоведение",
}
print(f"PDF-файлов: {len(PDF_FILES)}")
print(f"Источник: {PDF_DIR}")
print(f"Инструмент извлечения: {PDFTOTEXT}")

## 1. Извлечение текста и проверка страниц

`pdftotext -layout` сохраняет порядок строк таблиц. Удаляются невидимые служебные символы и заменяется частный bullet-глиф `U+F0B7`. Пустые страницы и страницы с менее чем 100 символами не индексируются, но остаются в отчёте. Это отсеивает два пустых листа обложки и обрезанный фрагмент таблицы в файле про оптику.

In [ ]:
def clean_page(raw_text: str) -> tuple[str, int, int, int]:
    private_bullets = raw_text.count("\uf0b7")
    text = unicodedata.normalize("NFC", raw_text).replace("\uf0b7", "•")
    text = text.replace("\xa0", " ").replace("\u00ad", "")
    invisible_pattern = r"[\u200b-\u200f\u202a-\u202e\u2060-\u206f\ufeff]"
    removed_invisible = len(re.findall(invisible_pattern, text))
    text = re.sub(invisible_pattern, "", text)
    lines = [re.sub(r"[ \t]+", " ", line).strip() for line in text.splitlines()]
    remaining_private_use = sum(unicodedata.category(char) == "Co" for char in text)
    return (
        "\n".join(line for line in lines if line),
        private_bullets,
        removed_invisible,
        remaining_private_use,
    )


def document_title(pdf_path: Path, pages: list[dict]) -> str:
    if pdf_path.name in MANUAL_TITLES:
        return MANUAL_TITLES[pdf_path.name]
    first_page = next(
        (page["text"] for page in pages if page["status"] == "indexed"), ""
    )
    match = re.search(r'[«“"]([^»”"]{4,160})[»”"]', first_page)
    if match:
        return re.sub(r"\s+", " ", match.group(1)).strip()
    return next(
        (line.strip() for line in first_page.splitlines() if len(line.strip()) > 8),
        pdf_path.stem,
    )


page_rows = []
audit_rows = []
for pdf_path in tqdm(PDF_FILES, desc="Извлекаю PDF-текст"):
    result = subprocess.run(
        [PDFTOTEXT, "-layout", "-enc", "UTF-8", str(pdf_path), "-"],
        capture_output=True,
        text=True,
        errors="replace",
        check=True,
    )
    raw_pages = result.stdout.split("\f")
    if raw_pages and not raw_pages[-1].strip():
        raw_pages.pop()  # pdftotext adds one form-feed after the last PDF page.

    document_pages = []
    duplicate_hashes = set()
    seen_hashes = set()
    omitted_pages = []
    replacement_chars = 0
    private_bullets = 0
    removed_invisible = 0
    remaining_private_use = 0

    for page_number, raw_page in enumerate(raw_pages, start=1):
        text, bullets, invisible, private_use = clean_page(raw_page)
        private_bullets += bullets
        removed_invisible += invisible
        remaining_private_use += private_use
        replacement_chars += text.count("\ufffd")
        status = "indexed"
        if not text:
            status = "empty_page"
        elif len(text) < MIN_PAGE_CHARACTERS:
            status = "omitted_short_fragment"

        if status != "indexed":
            omitted_pages.append(page_number)
        digest = hashlib.sha256(text.encode("utf-8")).hexdigest() if text else ""
        if digest and digest in seen_hashes:
            duplicate_hashes.add(digest)
        elif digest:
            seen_hashes.add(digest)

        row = {
            "document_name": pdf_path.name,
            "document_path": str(pdf_path.resolve()),
            "page_number": page_number,
            "status": status,
            "character_count": len(text),
            "text": text if status == "indexed" else "",
            "omitted_text_preview": " ".join(text.split())[:250]
            if status != "indexed"
            else "",
        }
        page_rows.append(row)
        document_pages.append(row)

    title = document_title(pdf_path, document_pages)
    for row in document_pages:
        row["document_title"] = title

    indexed_pages = [page for page in document_pages if page["status"] == "indexed"]
    warnings = []
    if omitted_pages:
        warnings.append(
            "Исключены пустые/короткие страницы: " + ", ".join(map(str, omitted_pages))
        )
    if replacement_chars:
        warnings.append(f"Символов замены U+FFFD: {replacement_chars}")
    if duplicate_hashes:
        warnings.append(f"Повторяющихся страниц: {len(duplicate_hashes)}")
    if remaining_private_use:
        warnings.append(
            f"Необработанных частных символов Unicode: {remaining_private_use}"
        )
    audit_rows.append(
        {
            "document_name": pdf_path.name,
            "document_title": title,
            "document_path": str(pdf_path.resolve()),
            "pages_total": len(raw_pages),
            "pages_indexed": len(indexed_pages),
            "pages_omitted": ", ".join(map(str, omitted_pages)),
            "characters_indexed": sum(
                page["character_count"] for page in indexed_pages
            ),
            "private_bullets_normalized": private_bullets,
            "remaining_private_use": remaining_private_use,
            "invisible_chars_removed": removed_invisible,
            "replacement_chars": replacement_chars,
            "duplicate_pages": len(duplicate_hashes),
            "quality_notes": "; ".join(warnings),
        }
    )

pages_df = pd.DataFrame(page_rows)
audit_df = pd.DataFrame(audit_rows)
PAGES_CSV = OUTPUT_DIR / "kubsu_docs_page_text.csv"
AUDIT_CSV = OUTPUT_DIR / "kubsu_docs_extraction_audit.csv"
pages_df.to_csv(PAGES_CSV, index=False, encoding="utf-8-sig")
audit_df.to_csv(AUDIT_CSV, index=False, encoding="utf-8-sig")

display(
    audit_df[
        [
            "document_name",
            "pages_total",
            "pages_indexed",
            "pages_omitted",
            "private_bullets_normalized",
            "replacement_chars",
            "duplicate_pages",
            "quality_notes",
        ]
    ]
)
print(f"\nПостраничный текст: {PAGES_CSV}")
print(f"Отчёт качества: {AUDIT_CSV}")

## 2. Ручная проверка извлечения

Откройте короткие фрагменты, исключённые из индекса, и несколько страниц с таблицами. `document_path` в CSV указывает на исходный PDF; страницы в `page_number` — исходные номера PDF, включая пустые страницы.

In [ ]:
display(
    pages_df.loc[
        pages_df.status.ne("indexed"),
        [
            "document_name",
            "page_number",
            "status",
            "character_count",
            "omitted_text_preview",
        ],
    ]
)

display(
    pages_df.loc[
        pages_df.status.eq("indexed"),
        ["document_name", "page_number", "document_title", "text"],
    ]
    .assign(preview=lambda frame: frame.text.str.slice(0, 600))[
        ["document_name", "page_number", "document_title", "preview"]
    ]
    .head(20)
)

assert pages_df.loc[pages_df.status.eq("indexed"), "text"].str.strip().ne("").all()
assert audit_df.replacement_chars.sum() == 0, (
    "Обнаружены повреждённые символы U+FFFD; проверьте эти страницы вручную."
)

## 3. Чанки USER2-base по страницам

Каждый чанк содержит название документа и исходную страницу. Это сохраняет ориентир для цитирования и PDF-ссылки, а реальное число токенов проверяется уже после добавления метаданных.

In [ ]:
client = QdrantClient(url=QDRANT_URL, timeout=30)
model = SentenceTransformer(MODEL_NAME)
tokenizer = model.tokenizer
VECTOR_SIZE = model.get_sentence_embedding_dimension()


def chunk_page(text: str, prefix: str) -> list[str]:
    if not tokenizer.is_fast:
        raise RuntimeError(
            "Для точного разбиения без повреждения текста нужен fast USER2 tokenizer"
        )

    prefix_tokens = len(tokenizer.encode(prefix, add_special_tokens=False))
    body_limit = CHUNK_SIZE_TOKENS - prefix_tokens
    if body_limit <= CHUNK_OVERLAP_TOKENS:
        raise ValueError(f"Метаданные слишком длинные для чанка: {prefix}")

    offsets = tokenizer(
        text,
        add_special_tokens=False,
        return_offsets_mapping=True,
    )["offset_mapping"]
    chunks = []
    start = 0
    while start < len(offsets):
        end = min(start + body_limit, len(offsets))
        while end > start:
            char_start = offsets[start][0]
            char_end = offsets[end - 1][1]
            body = text[char_start:char_end].strip()
            chunk = prefix + body
            if (
                len(tokenizer.encode(chunk, add_special_tokens=False))
                <= CHUNK_SIZE_TOKENS
            ):
                break
            end -= 1
        if end <= start:
            raise ValueError("Не удалось сформировать чанк в заданном токеновом лимите")
        if body:
            chunks.append(chunk)
        if end == len(offsets):
            break
        start = max(start + 1, end - CHUNK_OVERLAP_TOKENS)
    return chunks


records = []
for page in pages_df.loc[pages_df.status.eq("indexed")].itertuples(index=False):
    prefix = f"Документ: {page.document_title}. Страница {page.page_number}.\n"
    for chunk_index, text in enumerate(chunk_page(page.text, prefix)):
        token_count = len(tokenizer.encode(text, add_special_tokens=False))
        records.append(
            {
                "text": text,
                "chunk_index": chunk_index,
                "chunk_token_count": token_count,
                "document_name": page.document_name,
                "document_title": page.document_title,
                "document_path": page.document_path,
                "page_number": page.page_number,
                "record_id": f"{page.document_path}::page-{page.page_number}",
                "chat_name": page.document_title,
                "sender": "PDF-документ",
                "date": "",
                "file_paths": [page.document_path],
                "pdf_paths": [page.document_path],
                "source_type": "kubsu_pdf",
                "source_group": SOURCE_GROUP,
            }
        )

assert records, "Нет чанков для индексации"
assert max(record["chunk_token_count"] for record in records) <= CHUNK_SIZE_TOKENS
assert all("\ufffd" not in record["text"] for record in records), (
    "Чанк содержит символ повреждения U+FFFD"
)
indexed_page_count = int(pages_df.status.eq("indexed").sum())
print(f"Документов: {len(PDF_FILES)}; индексируемых страниц: {indexed_page_count}")
print(
    f"Чанков: {len(records):,}; максимум {max(r['chunk_token_count'] for r in records)} токенов"
)
display(
    pd.DataFrame(records)[
        ["document_name", "page_number", "chunk_index", "chunk_token_count", "text"]
    ].head(12)
)

## 4. Добавить документы в коллекцию

Загрузка добавляет только PDF-точки с меткой `source_group=kubsu_docs`. При повторном запуске новые точки сначала загружаются, затем удаляются устаревшие точки только этой группы; Telegram-переписка не удаляется.

In [ ]:
collection_names = {item.name for item in client.get_collections().collections}
if COLLECTION not in collection_names:
    client.create_collection(
        collection_name=COLLECTION,
        vectors_config=models.VectorParams(
            size=VECTOR_SIZE, distance=models.Distance.COSINE
        ),
    )
else:
    vector_config = client.get_collection(COLLECTION).config.params.vectors
    if isinstance(vector_config, dict):
        dimensions = {name: config.size for name, config in vector_config.items()}
        assert VECTOR_SIZE in dimensions.values(), (
            f"Размерность коллекции не совпадает с USER2-base: {dimensions}"
        )
    else:
        assert vector_config.size == VECTOR_SIZE, (
            f"Ожидалась размерность {VECTOR_SIZE}, получена {vector_config.size}"
        )

source_filter = models.Filter(
    must=[
        models.FieldCondition(
            key="source_type", match=models.MatchValue(value="kubsu_pdf")
        ),
        models.FieldCondition(
            key="source_group", match=models.MatchValue(value=SOURCE_GROUP)
        ),
    ]
)
old_ids = set()
offset = None
while True:
    old_points, offset = client.scroll(
        collection_name=COLLECTION,
        scroll_filter=source_filter,
        limit=256,
        offset=offset,
        with_payload=False,
        with_vectors=False,
    )
    old_ids.update(str(point.id) for point in old_points)
    if offset is None:
        break

new_ids = set()
for start in tqdm(range(0, len(records), BATCH_SIZE), desc="Векторизация и загрузка"):
    batch = records[start : start + BATCH_SIZE]
    vectors = model.encode(
        [record["text"] for record in batch],
        prompt_name="search_document",
        batch_size=BATCH_SIZE,
        show_progress_bar=False,
        convert_to_numpy=True,
    )
    points = []
    for record, vector in zip(batch, vectors):
        point_id = uuid.uuid5(
            uuid.NAMESPACE_URL,
            f"{record['document_path']}::{record['page_number']}::{record['chunk_index']}::{CHUNK_SIZE_TOKENS}::{CHUNK_OVERLAP_TOKENS}",
        )
        new_ids.add(str(point_id))
        points.append(
            models.PointStruct(id=str(point_id), vector=vector.tolist(), payload=record)
        )
    client.upsert(collection_name=COLLECTION, points=points, wait=True)

stale_ids = old_ids - new_ids
for start in range(0, len(stale_ids), 256):
    client.delete(
        collection_name=COLLECTION,
        points_selector=models.PointIdsList(
            points=list(stale_ids)[start : start + 256]
        ),
        wait=True,
    )

print(
    f"PDF-чанков загружено: {len(new_ids):,}; удалено устаревших PDF-чанков: {len(stale_ids):,}"
)
print(f"Всего точек в коллекции: {client.count(COLLECTION, exact=True).count:,}")
print("Пример PDF-ссылки:", records[0]["file_paths"][0])